# Albania Healthcare Resource Allocation Optimisation

**Companion project to the original Country Development K-Means university lab.**

## Decision question
**If Albania had an additional €100 million for health, how could it be allocated transparently across national programmes and the 12 prefectures?**

This is a **policy scenario-planning model**, not a claim that one mathematical allocation is clinically or politically optimal.

### Public evidence used
**INSTAT Population and Housing Census 2023**
- Resident population: 2,402,113
- Official prefecture-by-age table provides population and age structure for all 12 prefectures.
- National Census analysis reports disability prevalence of 6.5% among people aged 5+, with 59.5% of people with disability aged 65+.

Source:
https://www.instat.gov.al/media/14303/cens-2023.pdf

**WHO / World Bank health-financing indicators, 2023**
- Current health expenditure: 7.05% of GDP
- Domestic government health expenditure: 3.50% of GDP
- Government health expenditure: 12.28% of total government expenditure
- Out-of-pocket spending: 48.27% of current health expenditure
- Government financing: 49.60% of current health expenditure

Sources:
https://data.worldbank.org/indicator/SH.XPD.CHEX.GD.ZS?locations=AL
https://data.worldbank.org/indicator/SH.XPD.GHED.GD.ZS?locations=AL
https://data.worldbank.org/indicator/SH.XPD.GHED.GE.ZS?locations=AL
https://data.worldbank.org/indicator/SH.XPD.OOPC.CH.ZS?locations=AL
https://data.worldbank.org/indicator/SH.XPD.GHED.CH.ZS?locations=AL

WHO's Albania financial-protection review highlights outpatient medicines, primary-care coverage, low public investment, informal payments and gaps in protection from user charges as important priorities.

Source:
https://www.who.int/europe/publications/i/item/9789289055291

### What the project does
- builds a 12-prefecture demographic need table from official Census 2023 counts;
- preserves age and sex information;
- segments prefectures with K-Means for descriptive planning;
- creates an evidence-informed intervention score;
- uses constrained nonlinear optimisation with diminishing returns;
- allocates regional service-delivery funds using population + older-age burden;
- applies an equity floor so small prefectures are not starved of resources;
- compares need-adjusted allocation with population-only allocation;
- runs Monte Carlo sensitivity analysis;
- exports an intervention budget, regional budget and regional/intervention matrix.

> **Important:** weights, programme caps and the hypothetical €100m envelope are scenario assumptions. They are not observed causal treatment effects and require Ministry of Health, health-economics and clinical validation before real policy use.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Environment ready.")

## 1. National financing context

These are the latest 2023 values used in this scenario. They are stored explicitly so the notebook remains reproducible even if an external API changes.

In [ ]:
national = pd.Series({
    "Current health expenditure (% GDP)": 7.05,
    "Government health expenditure (% GDP)": 3.50,
    "Government health expenditure (% government expenditure)": 12.28,
    "Out-of-pocket share of current health expenditure (%)": 48.27,
    "Government share of current health expenditure (%)": 49.60,
    "Out-of-pocket expenditure per capita (US$)": 285.12
}, name="Albania 2023")

display(national.to_frame())

fig, ax = plt.subplots(figsize=(8.5,4.8))
pd.Series({
    "Out-of-pocket": national["Out-of-pocket share of current health expenditure (%)"],
    "Government": national["Government share of current health expenditure (%)"]
}).plot(kind="bar", ax=ax)
ax.set_title("Who finances current health spending? — Albania 2023")
ax.set_ylabel("Share of current health expenditure (%)")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 2. Build the official 2023 prefecture demographic table

Counts below come from INSTAT Census 2023 Table 6: **Resident population by sex, age group and prefecture**.

Age 65+ and 80+ are derived directly by summing the official age bands.

Sex is retained descriptively. The allocation model does **not** assume male or female residents deserve different funding; sex ratio is used only in descriptive segmentation.

In [ ]:
rows = [
    ["Berat",       140956,  69842,  71114, 11629,  8761,  5642,  4075, 2122],
    ["Dibër",       107178,  54051,  53127,  6780,  4648,  2744,  2319, 1377],
    ["Durrës",      226863, 113856, 113007, 15172, 10897,  6950,  5212, 2756],
    ["Elbasan",     232580, 116575, 116005, 16757, 12359,  7802,  5796, 3025],
    ["Fier",        240377, 120780, 119597, 20279, 14771,  9975,  7493, 4236],
    ["Gjirokastër",  60013,  30233,  29780,  5633,  4401,  2862,  2317, 1542],
    ["Korçë",       173091,  87295,  85796, 14129, 10777,  7249,  5599, 3140],
    ["Kukës",        61998,  31121,  30877,  2822,  1912,  1364,  1263,  788],
    ["Lezhë",        99384,  49665,  49719,  6876,  4885,  3413,  2694, 1701],
    ["Shkodër",     154479,  76392,  78087, 11377,  8282,  5846,  4343, 2591],
    ["Tiranë",      758513, 367428, 391085, 44411, 33403, 20753, 15279, 9372],
    ["Vlorë",       146681,  73210,  73471, 13026, 10613,  6487,  5129, 3248],
]

regional = pd.DataFrame(rows, columns=[
    "prefecture","population","male","female",
    "age_65_69","age_70_74","age_75_79","age_80_84","age_85_plus"
])

regional["age_65_plus"] = regional[
    ["age_65_69","age_70_74","age_75_79","age_80_84","age_85_plus"]
].sum(axis=1)
regional["age_80_plus"] = regional[["age_80_84","age_85_plus"]].sum(axis=1)

regional["age_65_plus_pct"] = regional["age_65_plus"] / regional["population"]
regional["age_80_plus_pct"] = regional["age_80_plus"] / regional["population"]
regional["sex_ratio_m_per_100_f"] = 100 * regional["male"] / regional["female"]

print("Population total:", f"{regional['population'].sum():,}")
print("Age 65+ total:", f"{regional['age_65_plus'].sum():,}")
print("Age 80+ total:", f"{regional['age_80_plus'].sum():,}")
display(regional)

In [ ]:
assert regional["population"].sum() == 2_402_113
assert (regional["male"] + regional["female"] == regional["population"]).all()
assert regional["age_65_plus"].sum() == 473_104

regional_view = regional[[
    "prefecture","population","age_65_plus","age_65_plus_pct",
    "age_80_plus","age_80_plus_pct","sex_ratio_m_per_100_f"
]].sort_values("age_65_plus_pct", ascending=False)

display(regional_view.style.format({
    "population":"{:,.0f}",
    "age_65_plus":"{:,.0f}",
    "age_65_plus_pct":"{:.2%}",
    "age_80_plus":"{:,.0f}",
    "age_80_plus_pct":"{:.2%}",
    "sex_ratio_m_per_100_f":"{:.1f}"
}))

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
plot_df = regional.sort_values("age_65_plus_pct", ascending=False)
ax.bar(plot_df["prefecture"], plot_df["age_65_plus_pct"]*100)
ax.set_title("Share of residents aged 65+ by prefecture — Census 2023")
ax.set_ylabel("Population aged 65+ (%)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9,6))
ax.scatter(
    regional["population"],
    regional["age_65_plus_pct"]*100,
    s=regional["age_80_plus_pct"]*3500
)
for _, r in regional.iterrows():
    ax.annotate(
        r["prefecture"],
        (r["population"], r["age_65_plus_pct"]*100),
        xytext=(5,4), textcoords="offset points", fontsize=8
    )
ax.set_title("Scale vs ageing pressure")
ax.set_xlabel("Resident population")
ax.set_ylabel("Population aged 65+ (%)")
plt.tight_layout()
plt.show()

## 3. Descriptive K-Means segmentation of prefecture need profiles

This clustering is **descriptive**, not a funding formula.

Features:
- population size;
- share aged 65+;
- share aged 80+;
- sex ratio.

K is chosen from 2–4 using silhouette score because there are only 12 prefectures.

In [ ]:
cluster_features = regional[[
    "population","age_65_plus_pct","age_80_plus_pct","sex_ratio_m_per_100_f"
]]
Z = StandardScaler().fit_transform(cluster_features)

cluster_scores = []
cluster_models = {}

for k in range(2,5):
    model = KMeans(n_clusters=k, n_init=30, random_state=RANDOM_STATE)
    labels = model.fit_predict(Z)
    score = silhouette_score(Z, labels)
    cluster_scores.append({"k":k,"silhouette":score})
    cluster_models[k] = model

cluster_scores = pd.DataFrame(cluster_scores)
best_k = int(cluster_scores.sort_values("silhouette",ascending=False).iloc[0]["k"])
regional["profile_cluster"] = cluster_models[best_k].labels_

display(cluster_scores)
print("Selected descriptive K:", best_k)

display(
    regional[[
        "prefecture","population","age_65_plus_pct",
        "age_80_plus_pct","sex_ratio_m_per_100_f","profile_cluster"
    ]].sort_values(["profile_cluster","age_65_plus_pct"],ascending=[True,False])
)

## 4. Define the €100m intervention portfolio

The scores below are **transparent scenario parameters**, not measured causal effects.

Each intervention receives three 0–1 scores:
- **policy priority** — strength of alignment with identified health-system gaps;
- **equity** — expected relevance to financially or clinically vulnerable people;
- **delivery readiness** — practical ability to absorb additional funding.

The objective uses diminishing returns: the first millions allocated to a high-priority programme are assumed to create more marginal value than later millions.

Minimum/maximum programme envelopes stop the optimiser from putting almost everything into one category.

In [ ]:
BUDGET_M = 100.0

interventions = pd.DataFrame({
    "intervention":[
        "Outpatient medicines & co-pay protection",
        "Primary care & diagnostics",
        "Workforce / rural access",
        "NCD prevention & screening",
        "Dental & mental-health access",
        "Data, quality & referral systems"
    ],
    "policy_priority":[1.00,0.95,0.85,0.80,0.65,0.55],
    "equity":[1.00,0.95,0.90,0.85,0.75,0.60],
    "delivery_readiness":[0.90,0.85,0.75,0.80,0.65,0.70],
    "min_m":[20,20,12,10,6,4],
    "max_m":[38,32,22,20,14,10]
})

OBJECTIVE_WEIGHTS = {
    "policy_priority":0.45,
    "equity":0.35,
    "delivery_readiness":0.20
}

interventions["composite_score"] = sum(
    interventions[col] * weight
    for col, weight in OBJECTIVE_WEIGHTS.items()
)

display(interventions)

## 5. Constrained nonlinear optimisation

Objective:

**maximise Σ scoreᵢ × √allocationᵢ**

subject to:
- total extra budget = €100m;
- each programme stays inside its declared minimum/maximum envelope.

The square-root utility introduces diminishing returns and avoids a trivial "all money to the highest score" solution.

In [ ]:
scores = interventions["composite_score"].to_numpy()
lower = interventions["min_m"].to_numpy(dtype=float)
upper = interventions["max_m"].to_numpy(dtype=float)

def optimise_budget(score_vector, total_budget=BUDGET_M):
    def objective(x):
        return -np.sum(score_vector * np.sqrt(np.maximum(x, 1e-9)))

    x0 = lower + (total_budget - lower.sum()) * (
        (upper-lower) / (upper-lower).sum()
    )

    result = minimize(
        objective,
        x0=x0,
        method="SLSQP",
        bounds=list(zip(lower,upper)),
        constraints=[{"type":"eq","fun":lambda x: x.sum()-total_budget}],
        options={"maxiter":1000,"ftol":1e-12}
    )

    if not result.success:
        raise RuntimeError(result.message)

    return result.x

optimal = optimise_budget(scores)
interventions["allocation_m"] = optimal
interventions["allocation_share"] = interventions["allocation_m"] / BUDGET_M

assert np.isclose(interventions["allocation_m"].sum(), BUDGET_M)

display(interventions[[
    "intervention","composite_score","allocation_m","allocation_share"
]].sort_values("allocation_m",ascending=False).style.format({
    "composite_score":"{:.3f}",
    "allocation_m":"€{:,.2f}m",
    "allocation_share":"{:.2%}"
}))

In [ ]:
fig, ax = plt.subplots(figsize=(10,5.5))
plot = interventions.sort_values("allocation_m")
ax.barh(plot["intervention"], plot["allocation_m"])
ax.set_title("Optimised hypothetical €100m intervention portfolio")
ax.set_xlabel("Allocation (€m)")
plt.tight_layout()
plt.show()

## 6. Regional need index

For regional service delivery, the model uses only official Census counts.

Raw need share:
- 50% population share — scale of service demand;
- 30% national share of people aged 65+ — older-care burden;
- 20% national share of people aged 80+ — higher-complexity ageing burden.

Then **15% of the regional pool is distributed equally across all 12 prefectures** as an equity floor. This prevents low-population areas from receiving almost nothing.

No disability-by-prefecture estimate is fabricated because the main Census publication provides national disability prevalence, not a comparable prefecture-level disability table for this model.

In [ ]:
regional["population_share"] = regional["population"] / regional["population"].sum()
regional["age65_share_national"] = regional["age_65_plus"] / regional["age_65_plus"].sum()
regional["age80_share_national"] = regional["age_80_plus"] / regional["age_80_plus"].sum()

REGIONAL_WEIGHTS = {
    "population_share":0.50,
    "age65_share_national":0.30,
    "age80_share_national":0.20
}
EQUITY_FLOOR_SHARE = 0.15

regional["need_share_raw"] = sum(
    regional[col] * weight
    for col,weight in REGIONAL_WEIGHTS.items()
)
regional["need_share"] = (
    EQUITY_FLOOR_SHARE * (1/len(regional))
    + (1-EQUITY_FLOOR_SHARE) * regional["need_share_raw"]
)

assert np.isclose(regional["need_share"].sum(),1.0)

display(regional[[
    "prefecture","population_share","age65_share_national",
    "age80_share_national","need_share"
]].sort_values("need_share",ascending=False).style.format({
    "population_share":"{:.2%}",
    "age65_share_national":"{:.2%}",
    "age80_share_national":"{:.2%}",
    "need_share":"{:.2%}"
}))

## 7. Split national programmes from regional service-delivery programmes

This scenario treats:
- **Outpatient medicines & co-pay protection** as a national reimbursement/protection envelope;
- **Data, quality & referral systems** as a national system envelope.

The remaining four programmes form the **regional service-delivery pool**.

In [ ]:
national_programmes = {
    "Outpatient medicines & co-pay protection",
    "Data, quality & referral systems"
}

interventions["delivery_level"] = np.where(
    interventions["intervention"].isin(national_programmes),
    "National",
    "Regional"
)

national_pool = interventions.loc[
    interventions["delivery_level"]=="National","allocation_m"
].sum()

regional_pool = interventions.loc[
    interventions["delivery_level"]=="Regional","allocation_m"
].sum()

print(f"National programme envelope: €{national_pool:,.2f}m")
print(f"Regional service-delivery pool: €{regional_pool:,.2f}m")
print(f"Check: €{national_pool + regional_pool:,.2f}m")

In [ ]:
regional["regional_allocation_m"] = regional["need_share"] * regional_pool
regional["eur_per_resident"] = (
    regional["regional_allocation_m"] * 1_000_000 / regional["population"]
)

# Population-only counterfactual with the same 15% equity floor.
regional["population_only_share"] = (
    EQUITY_FLOOR_SHARE * (1/len(regional))
    + (1-EQUITY_FLOOR_SHARE) * regional["population_share"]
)
regional["population_only_m"] = regional["population_only_share"] * regional_pool
regional["need_adjustment_m"] = (
    regional["regional_allocation_m"] - regional["population_only_m"]
)

regional_rank = regional[[
    "prefecture","population","age_65_plus_pct","age_80_plus_pct",
    "regional_allocation_m","eur_per_resident",
    "population_only_m","need_adjustment_m"
]].sort_values("regional_allocation_m",ascending=False)

display(regional_rank.style.format({
    "population":"{:,.0f}",
    "age_65_plus_pct":"{:.2%}",
    "age_80_plus_pct":"{:.2%}",
    "regional_allocation_m":"€{:,.2f}m",
    "eur_per_resident":"€{:,.2f}",
    "population_only_m":"€{:,.2f}m",
    "need_adjustment_m":"{:+.2f}m"
}))

In [ ]:
fig, ax = plt.subplots(figsize=(10,5.5))
p = regional.sort_values("regional_allocation_m")
ax.barh(p["prefecture"], p["regional_allocation_m"])
ax.set_title("Hypothetical regional service-delivery allocation")
ax.set_xlabel("Allocation (€m)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10,5.5))
p = regional.sort_values("need_adjustment_m")
ax.barh(p["prefecture"], p["need_adjustment_m"])
ax.axvline(0, linestyle="--")
ax.set_title("Age-need adjustment vs population-only allocation")
ax.set_xlabel("Difference (€m)")
plt.tight_layout()
plt.show()

## 8. Regional × intervention budget matrix

The four regional programme envelopes are distributed using the same prefecture need shares. This creates an auditable planning matrix rather than a single headline number.

In [ ]:
regional_programmes = interventions.loc[
    interventions["delivery_level"]=="Regional",
    ["intervention","allocation_m"]
].copy()

matrix = pd.DataFrame(index=regional["prefecture"])

for _, row in regional_programmes.iterrows():
    matrix[row["intervention"]] = (
        regional.set_index("prefecture")["need_share"] * row["allocation_m"]
    )

matrix["Regional total"] = matrix.sum(axis=1)

display(matrix.sort_values("Regional total",ascending=False).style.format("€{:,.2f}m"))

assert np.isclose(matrix["Regional total"].sum(), regional_pool)

## 9. Intervention sensitivity analysis

The optimiser should not be trusted if tiny changes in subjective weights completely change the answer.

We therefore sample **500 alternative policy-weight combinations** around the base priorities using a Dirichlet distribution and re-optimise the €100m portfolio each time.

In [ ]:
N_SIMS = 500
base_w = np.array([
    OBJECTIVE_WEIGHTS["policy_priority"],
    OBJECTIVE_WEIGHTS["equity"],
    OBJECTIVE_WEIGHTS["delivery_readiness"]
])

# Concentration controls how far scenarios move from the base weights.
alpha = base_w * 50

sim_allocations = []

for _ in range(N_SIMS):
    w = rng.dirichlet(alpha)
    scenario_scores = (
        interventions[["policy_priority","equity","delivery_readiness"]]
        .to_numpy() @ w
    )
    sim_allocations.append(optimise_budget(scenario_scores))

sim_allocations = np.asarray(sim_allocations)

sensitivity = pd.DataFrame({
    "intervention":interventions["intervention"],
    "p10_m":np.quantile(sim_allocations,0.10,axis=0),
    "median_m":np.quantile(sim_allocations,0.50,axis=0),
    "p90_m":np.quantile(sim_allocations,0.90,axis=0),
    "base_m":interventions["allocation_m"]
})

display(sensitivity.sort_values("base_m",ascending=False).style.format({
    "p10_m":"€{:,.2f}m",
    "median_m":"€{:,.2f}m",
    "p90_m":"€{:,.2f}m",
    "base_m":"€{:,.2f}m"
}))

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
s = sensitivity.sort_values("base_m")
y = np.arange(len(s))
ax.errorbar(
    s["median_m"], y,
    xerr=[
        s["median_m"]-s["p10_m"],
        s["p90_m"]-s["median_m"]
    ],
    fmt="o", capsize=4
)
ax.set_yticks(y, s["intervention"])
ax.set_title("Intervention-allocation sensitivity (10th–90th percentile)")
ax.set_xlabel("Allocation (€m)")
plt.tight_layout()
plt.show()

## 10. Regional weighting sensitivity

A second simulation varies:
- the population / age 65+ / age 80+ weights;
- the equity-floor share.

This tests whether the prefecture ranking is robust to reasonable changes in the demographic funding formula.

In [ ]:
N_REGION_SIMS = 1000
base_rw = np.array([0.50,0.30,0.20])
regional_alpha = base_rw * 60

regional_sim = []

base_vectors = regional[
    ["population_share","age65_share_national","age80_share_national"]
].to_numpy()

for _ in range(N_REGION_SIMS):
    w = rng.dirichlet(regional_alpha)
    floor = rng.uniform(0.10,0.20)
    raw_share = base_vectors @ w
    share = floor*(1/len(regional)) + (1-floor)*raw_share
    regional_sim.append(share * regional_pool)

regional_sim = np.asarray(regional_sim)

regional_sensitivity = pd.DataFrame({
    "prefecture":regional["prefecture"],
    "p10_m":np.quantile(regional_sim,0.10,axis=0),
    "median_m":np.quantile(regional_sim,0.50,axis=0),
    "p90_m":np.quantile(regional_sim,0.90,axis=0),
    "base_m":regional["regional_allocation_m"]
})

display(
    regional_sensitivity.sort_values("base_m",ascending=False)
    .style.format({
        "p10_m":"€{:,.2f}m",
        "median_m":"€{:,.2f}m",
        "p90_m":"€{:,.2f}m",
        "base_m":"€{:,.2f}m"
    })
)

## 11. Priority dashboard

The key insight is not just who gets the most money.

- **Tiranë** has the largest absolute requirement because it contains 31.6% of Albania's population.
- Older prefectures can receive **more per resident** because age-related service need is weighted explicitly.
- The equity floor ensures small prefectures retain a minimum service-development envelope.

In [ ]:
priority_dashboard = regional[[
    "prefecture","population","age_65_plus_pct","age_80_plus_pct",
    "regional_allocation_m","eur_per_resident","need_adjustment_m",
    "profile_cluster"
]].copy()

priority_dashboard["absolute_rank"] = (
    priority_dashboard["regional_allocation_m"]
    .rank(method="dense",ascending=False).astype(int)
)
priority_dashboard["per_capita_rank"] = (
    priority_dashboard["eur_per_resident"]
    .rank(method="dense",ascending=False).astype(int)
)

display(
    priority_dashboard.sort_values("absolute_rank")
    .style.format({
        "population":"{:,.0f}",
        "age_65_plus_pct":"{:.2%}",
        "age_80_plus_pct":"{:.2%}",
        "regional_allocation_m":"€{:,.2f}m",
        "eur_per_resident":"€{:,.2f}",
        "need_adjustment_m":"{:+.2f}m"
    })
)

## 12. Export decision artefacts

In [ ]:
interventions.to_csv("albania_health_intervention_budget.csv", index=False)
regional.to_csv("albania_health_prefecture_budget.csv", index=False)
matrix.to_csv("albania_health_prefecture_intervention_matrix.csv")
sensitivity.to_csv("albania_health_intervention_sensitivity.csv", index=False)
regional_sensitivity.to_csv("albania_health_prefecture_sensitivity.csv", index=False)

print("Exported:")
print("- albania_health_intervention_budget.csv")
print("- albania_health_prefecture_budget.csv")
print("- albania_health_prefecture_intervention_matrix.csv")
print("- albania_health_intervention_sensitivity.csv")
print("- albania_health_prefecture_sensitivity.csv")

## 13. Executive summary

In [ ]:
top_absolute = regional.sort_values("regional_allocation_m",ascending=False).iloc[0]
top_per_capita = regional.sort_values("eur_per_resident",ascending=False).iloc[0]
medicine_budget = interventions.loc[
    interventions["intervention"]=="Outpatient medicines & co-pay protection",
    "allocation_m"
].iloc[0]
primary_budget = interventions.loc[
    interventions["intervention"]=="Primary care & diagnostics",
    "allocation_m"
].iloc[0]

print(f"""
ALBANIA HEALTHCARE RESOURCE-ALLOCATION SCENARIO
------------------------------------------------
Hypothetical additional budget: €{BUDGET_M:.0f}m

2023 FINANCING CONTEXT
Current health expenditure: {national['Current health expenditure (% GDP)']:.2f}% of GDP
Government health expenditure: {national['Government health expenditure (% GDP)']:.2f}% of GDP
Out-of-pocket share: {national['Out-of-pocket share of current health expenditure (%)']:.2f}%

INTERVENTION SCENARIO
Medicines / co-pay protection: €{medicine_budget:.2f}m
Primary care / diagnostics: €{primary_budget:.2f}m
National programme envelope: €{national_pool:.2f}m
Regional service-delivery pool: €{regional_pool:.2f}m

REGIONAL SCENARIO
Largest absolute regional allocation: {top_absolute['prefecture']} — €{top_absolute['regional_allocation_m']:.2f}m
Highest regional allocation per resident: {top_per_capita['prefecture']} — €{top_per_capita['eur_per_resident']:.2f}
Oldest prefecture by 65+ share: {regional.sort_values('age_65_plus_pct',ascending=False).iloc[0]['prefecture']}
National population aged 65+: {regional['age_65_plus'].sum():,} ({regional['age_65_plus'].sum()/regional['population'].sum():.2%})

METHOD
- constrained diminishing-return optimisation for interventions
- Census-based demographic need index for prefectures
- 15% equal-share equity floor
- descriptive K-Means segmentation
- 500 intervention sensitivity scenarios
- 1,000 regional weighting sensitivity scenarios

This is a transparent POLICY SCENARIO, not a causal claim that these exact
amounts would maximise health outcomes. Real deployment would require disease
burden, workforce capacity, facility access, medicine utilisation, poverty,
travel-time, waiting-list and programme cost-effectiveness data.
""")

## 14. What would make this production-grade?

The most valuable next data additions would be:

1. **Prefecture / municipality disease burden** — cardiovascular disease, diabetes, cancer, respiratory disease, maternal health.
2. **Health workforce supply** — GPs, specialists, nurses and pharmacists per 10,000 residents.
3. **Travel-time / access** — time to primary-care centres, hospitals and diagnostic services.
4. **Medicine affordability and utilisation** — reimbursement, co-payments and prescription demand by region/income.
5. **Poverty / household financial risk** — so financial-protection funds can be explicitly targeted.
6. **Waiting lists and referral flows** — to identify capacity bottlenecks.
7. **Programme-level cost-effectiveness evidence** — DALYs/QALYs, avoided admissions, detection rates and adherence.

With those data, the optimisation objective could move from transparent scenario scores to **measured expected health gain, financial protection and equity outcomes**.

### Interview defence
> “I kept my original university K-Means country-development lab unchanged, then built a separate decision-analytics extension. I used official INSTAT Census 2023 age/sex counts and latest WHO/World Bank health-financing indicators, formulated a constrained diminishing-return budget optimiser, distributed regional service funds using transparent demographic need weights with an equity floor, and stress-tested both the intervention and prefecture allocations. I explicitly separate evidence from assumptions so the model is defensible rather than pretending scenario weights are causal effects.”